<img src=../images/gdd-logo.png width=300px align=right>

# Transform

In principle using `.groupby()` and then merging our aggregates back to our original dataframe is quite common. Maybe you want to group by some information (things like say, average session length) and add this information to a raw dataset. 

To perform the aggregation first makes sense, but especially for large dataframes the join/merge operations that follow can be a bit expensive. There is an alternative.

In this section we will cover:

* [Overview of `transform()`](#overview)
* [<mark>Exercise: Use `transform()` and `assign()` to create new columns</mark>](#exercise)

Before we do anything though let's again import pandas read in our data.

In [ ]:
import pandas as pd

chickweight = (
    pd.read_csv('../data/chickweight.csv') 
    .rename(str.lower, axis='columns')
)



<a id='overview'></a>

## Overview of `transform()`

<img src="../images/07_Transform/alternative.png" width="140" height="140" align="center"/>

The `.transform()` method allows us to do aggregation, as well as the join/merge in one go. 

To demonstrate how this works, let us imagine we want to add the mean chickweight per diet to the dataframe.

In [3]:
mean_weight_per_diet = (
    chickweight
    .groupby("diet")['weight']
    .mean()
)
mean_weight_per_diet

diet
1    102.645455
2    122.616667
3    142.950000
4    135.262712
Name: weight, dtype: float64

The problem here is that aggregating produces a dataframe which is a length to our original dataframe. 

In [ ]:
chickweight.shape, mean_weight_per_diet.shape

This is why previously we used merge or join to combine this data.

In [ ]:
(
    chickweight
    .merge(mean_weight_per_diet, on='diet', suffixes=('','_mean'))
)

However if we use the the `.transform()` method, we can calculate the mean chick weight information and add it to the chickweight datafame in one go.

When we use transform the the values corespond to the different diets.

In [ ]:
(
    chickweight
    .groupby("diet")['weight']
    .transform('mean')
)

Importantly, the output is the same length as the original dataframe and 

In [ ]:
(
    chickweight
    .groupby("diet")['weight']
    .transform('mean')
).shape

This means we can easily add this information as a new column in one go with the `.assign()` method.

In [4]:
(
    chickweight
    .assign(mean_weight_diet = lambda df: df.groupby("diet")['weight'].transform('mean'))
)

,rownum,weight,time,chick,diet,mean_weight_diet
0,1,42,0,1,1,102.645455
1,2,51,2,1,1,102.645455
2,3,59,4,1,1,102.645455
3,4,64,6,1,1,102.645455
4,5,76,8,1,1,102.645455
...,...,...,...,...,...,...
573,574,175,14,50,4,135.262712
574,575,205,16,50,4,135.262712
575,576,234,18,50,4,135.262712
576,577,264,20,50,4,135.262712


<a id='exercise'></a>
## <mark>Exercises: Use `.assign()` and `transform()` to create new columns</mark>

### Exercise 1

Take the original `chickweight` dataframe and create these columns on the raw data without performing a join: 

1. **mean_weight_diet**: which calculates the mean weight per diet 
2. **mean_weight_diet_time**: which calculates the mean weight per diet at a given time

**BONUS:**

3. **num_chickens_diet**: which calculates the total number of chickens per diet - explore what the `.nunique()` method does to do this.

In [13]:
#1.
(
    chickweight
    .assign(
        mean_weight_per_diet = lambda df: df.groupby(['diet'])['weight'].transform('mean'),
        mean_weight_per_diet_time = lambda df: df.groupby(['diet', 'time'])['weight'].transform('mean'),
        chickens_per_diet = lambda df: df.groupby(['diet'])['chick'].transform('nunique')
    )
)

,rownum,weight,time,chick,diet,mean_weight_per_diet,mean_weight_per_diet_time,chickens_per_diet
0,1,42,0,1,1,102.645455,41.400000,20
1,2,51,2,1,1,102.645455,47.250000,20
2,3,59,4,1,1,102.645455,56.473684,20
3,4,64,6,1,1,102.645455,66.789474,20
4,5,76,8,1,1,102.645455,79.684211,20
...,...,...,...,...,...,...,...,...
573,574,175,14,50,4,135.262712,161.800000,10
574,575,205,16,50,4,135.262712,182.000000,10
575,576,234,18,50,4,135.262712,202.900000,10
576,577,264,20,50,4,135.262712,233.888889,10


In [14]:
# %load ../answers/07_Transform/ex-transforms.py
(
    chickweight
    .assign(mean_weight_diet=lambda df: df.groupby("diet")['weight'].transform('mean'),
            mean_weight_diet_time=lambda df: df.groupby(["diet", "time"])['weight'].transform('mean'),
            num_chickens_diet=lambda df: df.groupby("diet")["chick"].transform(lambda x: x.nunique())
           )
)


### Exercise 2: Find the fattest chicken per diet

Do you rember in the last notebook we tried to find the fattest chicken per diet?

Is it any easier now we know about the `transform` method?

In [ ]:
#yes

NameError: name 'yes' is not defined

In [18]:
# %load ../answers/07_Transform/ex-fattest-chick-transform.py
(
    pd.merge(
        chickweight,
        chickweight.groupby('diet')['weight'].transform('max').rename('max_weight'),
        left_index=True,
        right_index=True
    )
    .loc[lambda df: df['weight']==df['max_weight']]
)

,rownum,weight,time,chick,diet,max_weight
83,84,305,21,7,1,305
231,232,331,21,21,2,331
399,400,373,21,35,3,373
553,554,322,21,48,4,322
